In [ ]:
!nvidia-smi

In [ ]:
#%pip install tweet-preprocessor
%pip install -U spacy spacy-curated-transformers
!python -m spacy download en_core_web_trf
%pip install -U spacy[cuda13.0]

import json
import os

import preprocessor as p
import spacy
from tqdm import tqdm

In [ ]:
def clean_text(content):
    """Cleans text content by removing unwanted characters and formatting issues."""
    content = str(content)
    content = re.sub(r'<[^>]+>', '', content)
    content = re.sub(r'[\t\n\r\f\v]+', ' ', content)
    content = re.sub(r' {2,}', ' ', content)
    try:
        content = content.split(":", 1)[1].strip()
    except:
        pass
    content = p.clean(content)
    return content.strip()

In [ ]:
path = "/content/"
files = [path + f for f in os.listdir(path) if f.endswith(".txt")]
files[0]

In [ ]:
docs = []
for f in files:
    with open(f, 'r') as file_object:
        sentences = file_object.read().split("\n")
        sentence = ""
        for s in sentences:
            sentence += clean_text(s)
        docs.append(clean_text(sentence))
len(docs)

## NER

In [ ]:
#entities = {}
#for idx, sentence in enumerate(docs):
#    tokens = word_tokenize(sentence)
#    pos_tags = pos_tag(tokens)
#    named_entities = ne_chunk(pos_tags)

#    for chunk in named_entities:
#        if isinstance(chunk, Tree):
#            entity_label = chunk.label()
#            entity_text = " ".join(c[0] for c in chunk)

#            entities.setdefault(entity_label, set()).add(entity_text)

#serializable_entities = {label: list(entities[label]) for label in entities}
#print(json.dumps(serializable_entities, indent=4))

In [ ]:
spacy.require_gpu()
print(spacy.prefer_gpu())
nlp = spacy.load("en_core_web_trf")

PEOPLE = {"PERSON"}
ORG = {"NORP", "ORG"}
EVENTS = {"EVENT"}
PLACES = {"FAC", "GPE", "LOC"}
OTHER = {"LAW", "LANGUAGE"}

people, org, events, places, other = set(), set(), set(), set(), set()
for text in tqdm(docs):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in PEOPLE:
            people.add(ent.text)
        elif ent.label_ in ORG:
            org.add(ent.text)
        elif ent.label_ in EVENTS:
            events.add(ent.text)
        elif ent.label_ in PLACES:
            places.add(ent.text)
        elif ent.label in OTHER:
            other.add(ent.text)

entities = {
    "PEOPLE": people,
    "ORG": org,
    "EVENTS": events,
    "PLACES": places,
    "OTHER": other,
}

serializable_entities = {label: list(entities[label]) for label in entities}
print(json.dumps(serializable_entities, indent=5))

## Topic Extraction

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)
model.to(device)
model.eval()

In [ ]:
import re
import torch

SYSTEM_MSG = "You are a political debate topic classifier. Follow the user's format exactly."

USER_TEMPLATE = """Task:
Identify the main topics discussed in this debate.

Return ONLY the topic names separated by commas in this exact format:
<TOPICS: topic1, topic2, ...>

Do not output anything else.

Debate:
{chunk}
"""

In [ ]:
def chunk_by_tokens(text: str, tokenizer, max_input_tokens: int):
    ids = tokenizer.encode(text, add_special_tokens=False)
    for i in range(0, len(ids), max_input_tokens):
        chunk_ids = ids[i:i + max_input_tokens]
        yield tokenizer.decode(chunk_ids)


def extract_topics(raw: str):
    TOPIC_RE = re.compile(r"TOPICS\s*:\s*(.+)", re.IGNORECASE)
    m = TOPIC_RE.search(raw)
    if not m:
        return []
    items = [t.replace("-", " ").replace("_", " ").strip() for t in m.group(1).split(",")]
    # normalize + drop empties
    return [t for t in (i.lower().replace(" ", "_") for i in items) if t]

In [ ]:
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 128

all_topics = set()
for debate in docs:
    for chunk in chunk_by_tokens(debate, tokenizer, MAX_INPUT_TOKENS):
        user_prompt = USER_TEMPLATE.format(chunk=chunk)

        messages = [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": user_prompt},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.0,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )

        gen_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        raw = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

        topics = extract_topics(raw)
        all_topics.update(topics)

print("\nFINAL TOPICS:", sorted(all_topics))

In [ ]:
entities['TOPICS'] = all_topics

In [ ]:
import json

with open('topic_results.json', 'w') as f:
    json.dump(
        entities,
        f,
        default=lambda o: sorted(o) if isinstance(o, set) else str(o),
        ensure_ascii=False,
        indent=2
    )